# Super-Human Chess AI Training (Kaggle Edition)
This notebook trains the DeepVision10M model on the full chess dataset using performance optimizations like Lazy Loading, Automatic Mixed Precision, and `torch.compile`.

In [ ]:
!pip install python-chess tqdm pandas numpy torch

In [ ]:
import torch
import torch.nn as nn
import chess
import numpy as np
import time

# --- 1. THE DEEPVISION ELITE ARCHITECTURE ---
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class MultiScaleBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, in_channels // 4, kernel_size=1)
        self.conv3x3 = nn.Conv2d(in_channels, in_channels // 2, kernel_size=3, padding=1)
        self.conv5x5 = nn.Conv2d(in_channels, in_channels // 4, kernel_size=5, padding=2)
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.GELU()
        self.se = SEBlock(in_channels)
    def forward(self, x):
        out = torch.cat([self.conv1x1(x), self.conv3x3(x), self.conv5x5(x)], dim=1)
        return self.relu(self.se(self.bn(out)) + x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, embed_dim * 4), nn.GELU(), nn.Linear(embed_dim * 4, embed_dim))
        self.ln2 = nn.LayerNorm(embed_dim)
        self.spatial_bias = nn.Parameter(torch.zeros(1, num_heads, 64, 64))
    def forward(self, x):
        residual = x
        x = self.ln1(x)
        b, n, d = x.shape
        q, k, v = [x.view(b, n, self.num_heads, self.head_dim).transpose(1, 2) for _ in range(3)]
        attn = torch.softmax(((q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)) + self.spatial_bias, dim=-1)
        x = residual + (attn @ v).transpose(1, 2).contiguous().view(b, n, d)
        return x + self.ffn(self.ln2(x))

class ReasoningLayer(nn.Module):
    def __init__(self, embed_dim, num_heads=8):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(nn.Linear(embed_dim, embed_dim * 2), nn.GELU(), nn.Linear(embed_dim * 2, embed_dim))
    def forward(self, spatial_features, evaluation_context):
        attn_out, _ = self.cross_attn(spatial_features, evaluation_context, evaluation_context)
        x = self.norm(spatial_features + attn_out)
        return x + self.mlp(x)

class MoEPolicyHead(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.experts = nn.ModuleList([nn.Sequential(nn.Conv2d(in_channels, 64, 1), nn.BatchNorm2d(64), nn.GELU(), nn.Flatten(), nn.Linear(64*8*8, 4096)) for _ in range(3)])
        self.router = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_channels, 32), nn.GELU(), nn.Linear(32, 3), nn.Softmax(dim=-1))
    def forward(self, x):
        weights = self.router(x)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1)
        return torch.bmm(weights.unsqueeze(1), expert_outputs).squeeze(1)

class DeepVisionElite(nn.Module):
    def __init__(self, refinement_steps=3):
        super().__init__()
        self.refinement_steps = refinement_steps
        self.input_conv = nn.Sequential(nn.Conv2d(32, 256, kernel_size=3, padding=1, bias=False), nn.BatchNorm2d(256), nn.GELU())
        self.blocks = nn.Sequential(*[MultiScaleBlock(256) for _ in range(20)])
        self.attention = nn.ModuleList([TransformerBlock(256, 16) for _ in range(4)])
        self.reasoner = ReasoningLayer(256, num_heads=16)
        self.refinement_mlp = nn.Sequential(nn.Linear(256, 256), nn.GELU(), nn.Linear(256, 256))
        
        # Output Heads
        self.policy_head = MoEPolicyHead(256)
        self.mirror_head = MoEPolicyHead(256) 
        self.dsi_head = nn.Sequential(nn.Conv2d(256, 32, 1), nn.BatchNorm2d(32), nn.GELU(), nn.Flatten(), nn.Linear(32*8*8, 4096))
        self.value_head = nn.Sequential(nn.Conv2d(256, 8, 1), nn.BatchNorm2d(8), nn.GELU(), nn.Flatten(), nn.Linear(8*8*8, 256), nn.GELU(), nn.Linear(256, 1), nn.Tanh())
        self.lookahead_head = nn.Sequential(nn.Conv2d(256, 32, 1), nn.BatchNorm2d(32), nn.GELU(), nn.Flatten(), nn.Linear(32*8*8, 512), nn.GELU(), nn.Linear(512, 256))
        self.aux_head = nn.Sequential(nn.Conv2d(256, 4, 1), nn.BatchNorm2d(4), nn.GELU(), nn.Flatten(), nn.Linear(4*8*8, 128), nn.GELU(), nn.Linear(128, 2))

    def forward(self, x):
        x = self.input_conv(x)
        x = self.blocks(x)
        b, c, h, w = x.shape
        x_flat = x.view(b, c, h * w).permute(0, 2, 1)
        for layer in self.attention: x_flat = layer(x_flat)
        for _ in range(self.refinement_steps):
            pooled = torch.mean(x_flat, dim=1, keepdim=True)
            x_flat = x_flat + self.refinement_mlp(self.reasoner(x_flat, pooled))
        x_final = x_flat.permute(0, 2, 1).view(b, c, h, w)
        return self.policy_head(x_final), self.value_head(x_final), self.aux_head(x_final), self.lookahead_head(x_final), self.dsi_head(x_final), self.mirror_head(x_final)

# --- 2. ENCODING UTILITIES ---
def board_to_tensor_elite(board):
    tensor = np.zeros((32, 8, 8), dtype=np.float32)
    for color in [chess.WHITE, chess.BLACK]:
        for pt in range(1, 7):
            idx = (0 if color == chess.WHITE else 6) + (pt - 1)
            for sq in board.pieces(pt, color): tensor[idx, divmod(sq, 8)] = 1.0
            idx += 12
            for sq in board.pieces(pt, color):
                for a_sq in board.attacks(sq): tensor[idx, divmod(a_sq, 8)] = 1.0
    tensor[24, :, :] = 1.0 if board.turn == chess.WHITE else 0.0
    if board.has_kingside_castling_rights(chess.WHITE): tensor[25, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.WHITE): tensor[26, :, :] = 1.0
    if board.has_kingside_castling_rights(chess.BLACK): tensor[27, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.BLACK): tensor[28, :, :] = 1.0
    tensor[29, :, :] = board.halfmove_clock / 100.0
    if board.ep_square: tensor[30, divmod(board.ep_square, 8)] = 1.0
    tensor[31, :, :] = min(board.fullmove_number, 100) / 100.0
    return torch.from_numpy(tensor).float()

# --- 3. BATCHED MCTS ENGINE ---
class MCTSNode:
    def __init__(self, board, parent=None, move=None, prior=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.prior = prior
        self.children = {}
        self.visit_count = 0
        self.value_sum = 0
        self.is_expanded = False

    def value(self):
        return self.value_sum / self.visit_count if self.visit_count > 0 else 0

    def select_child(self, c_puct=1.4):
        best_score, best_child = -float('inf'), None
        for move, child in self.children.items():
            u_score = c_puct * child.prior * np.sqrt(self.visit_count) / (1 + child.visit_count)
            score = child.value() + u_score
            if score > best_score: best_score, best_child = score, child
        return best_child

class MCTSSearcher:
    def __init__(self, model, device):
        self.model = model
        self.device = device

    def search(self, board, max_time=3.0, batch_size=64):
        """Used by app.py: Searches for the best move within a time limit."""
        root = MCTSNode(board.copy())
        start_time = time.time()
        self._expand_batch([root])
        
        while time.time() - start_time < max_time:
            self._run_batch_iteration(root, batch_size, max_time, start_time)
            
        if not root.children: return list(board.legal_moves)[0]
        return max(root.children.items(), key=lambda x: x[1].visit_count)[0]

    def search_for_self_play(self, board, simulations=200, batch_size=64):
        """Used by self_play.py: Searches for a fixed number of nodes, returns the root."""
        root = MCTSNode(board.copy())
        self._expand_batch([root])
        
        nodes_evaluated = 1
        while nodes_evaluated < simulations:
            leaves_expanded = self._run_batch_iteration(root, batch_size)
            nodes_evaluated += leaves_expanded
            
        return root

    def _run_batch_iteration(self, root, batch_size, max_time=None, start_time=None):
        leaves_to_expand = []
        while len(leaves_to_expand) < batch_size:
            if max_time and (time.time() - start_time >= max_time): break
                
            node = root
            while node.is_expanded and node.children:
                node = node.select_child()
                
            if node.board.is_game_over():
                res = node.board.result()
                val = 1.0 if res == "1-0" else (-1.0 if res == "0-1" else 0)
                self._backpropagate(node, val)
            elif node not in leaves_to_expand:
                leaves_to_expand.append(node)
                
        if leaves_to_expand:
            self._expand_batch(leaves_to_expand)
        return len(leaves_to_expand)

    def _expand_batch(self, nodes):
        tensors = torch.stack([board_to_tensor_elite(n.board) for n in nodes]).to(self.device)
        with torch.no_grad():
            with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu', dtype=torch.bfloat16):
                p_out, v_out, _, _, dsi_out, mirror_out = self.model(tensors)
                
            for i, node in enumerate(nodes):
                # Add Dirichlet Noise at the root node for exploration during self-play
                noise = 0
                if node.parent is None: 
                    noise = np.random.dirichlet([0.3] * 4096)
                
                final_logic = p_out[i] + 0.4 * dsi_out[i] - 0.1 * mirror_out[i]
                policy = torch.softmax(final_logic.float(), dim=0).cpu().numpy()
                
                if node.parent is None:
                    policy = 0.75 * policy + 0.25 * noise
                    
                value = v_out[i].float().item()
                ordered_moves = self._get_ordered_moves(node.board)
                
                for move in ordered_moves:
                    idx = move.from_square * 64 + move.to_square
                    child_board = node.board.copy()
                    child_board.push(move)
                    node.children[move] = MCTSNode(child_board, parent=node, move=move, prior=policy[idx])
                
                node.is_expanded = True
                self._backpropagate(node, value)

    def _backpropagate(self, node, value):
        while node:
            node.visit_count += 1
            node.value_sum += value
            node = node.parent
            value = -value

    def _get_ordered_moves(self, board):
        def score(move):
            s = 0
            if board.is_capture(move): s += 10
            if board.gives_check(move): s += 5
            if move.promotion: s += 8
            return s
        return sorted(list(board.legal_moves), key=score, reverse=True)